# OS3 AI Engineer Evaluation – Set 2
## L2-01 Retrieval-Augmented Generation (RAG) Chatbot

This notebook covers PDF/TXT ingestion, chunking, embeddings, Chroma vector storage, retrieval, and context-aware generation.

In [ ]:
%pip install -q langchain langchain-openai langchain-community langchain-text-splitters chromadb pypdf python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is not set.")

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate

DOCUMENT_PATH = "sample.pdf"
if not os.path.exists(DOCUMENT_PATH):
    raise FileNotFoundError("Upload a PDF/TXT to Jupyter and set DOCUMENT_PATH.")

loader = PyPDFLoader(DOCUMENT_PATH) if DOCUMENT_PATH.lower().endswith(".pdf") else TextLoader(DOCUMENT_PATH, encoding="utf-8")
documents = loader.load()
chunks = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120).split_documents(documents)
print("Documents:", len(documents), "Chunks:", len(chunks))


In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(chunks, embeddings, collection_name="os3_rag_demo")
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_prompt = ChatPromptTemplate.from_template(
'''Answer the question using only the supplied context.
If the answer is not present, say the document does not contain enough information.

Context:
{context}

Question:
{question}'''
)

def rag_query(question):
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)
    answer = llm.invoke(rag_prompt.format_messages(context=context, question=question)).content
    return answer, docs


In [ ]:
answer, sources = rag_query("What is the main topic of this document?")
print(answer)
print("Retrieved chunks:", len(sources))

## RAG pipeline
Ingestion → Chunking → Embedding → Vector Store → Retrieval → Generation

Example questions:
- What is the purpose of this document?
- List the main requirements.
- What important dates are mentioned?